# 02. Next greater

**Tier:** Deep. **Why you'd use this:** For each day's reading you want the next strictly warmer day to its right. Scanning rightward from every index repeats work. A stack of waiting indexes answers each day once.

**After this lesson you can:** keep indexes whose next greater value is still unknown; resolve them when a larger value arrives.
About 25 minutes. **Uses:** [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb), [Stacks](01-backspace-lesson.ipynb)


## Quick review

1. What does a stack store in the backspace problem?
2. Why store indexes instead of values when the answer is "next greater to the right"?

<details><summary>Answers</summary>

Kept characters. Indexes let you write into the answer slot for an earlier position when a later greater value shows up. Skim [Stacks](01-backspace-lesson.ipynb) if push/pop feels shaky.
</details>


## Recognize it

For each index you need the nearest value to its right that is strictly greater, or a sentinel when none exists. The prompt says "next greater element", "warmer day", or "first larger to the right".

Signals: each position waits for a future value that beats it; later values can resolve many earlier waiters. A look-alike that is not this pattern: next greater in a circular array needs a second lap — related, but the wrap is an extra rule.


## Why it works this way

The nested scan starts at each index and walks right until it finds a greater value. In the worst case every walk goes to the end, so comparisons grow like `n²`. The stack approach stores indexes that are still waiting, in decreasing order of their values. When the current value is greater than the value at the top index, that waiter has found its answer: pop and write. Repeat until the stack's top is no longer smaller. Then push the current index.

Each index is pushed once and popped at most once. Total stack operations are proportional to `n`, even though an inner `while` looks nested. That is the usual amortization for monotonic stacks. Extra memory is the stack and the answer list — both grow with `n`.

The invariant: values at stacked indexes are in decreasing order from bottom to top. A new value that is not greater than the top cannot help anyone under the top yet, so it just waits. Using `<=` instead of `<` would treat equals as "greater" and give wrong answers when the prompt asks for strictly greater.

Temperatures and stock spans are the same skeleton with different payloads. Circular next-greater doubles the walk with modulo indexes so the front can see values that wrapped — same pops, longer pass.
Daily temperatures ask for the number of days until a warmer day, not the warmer value itself. The stack still stores indexes; the answer slot stores `later_index - earlier_index` instead of `nums[later]`. Same pops, different payload. Next-smaller is the dual: keep an increasing stack and pop when a stricter smaller value arrives.

If the prompt allows equals to count, switch `<` to `<=` deliberately and say so in the invariant. Silent use of the wrong relational operator is the most common off-by-one style bug in this family. Circular arrays need a second logical pass with `index % n` so values at the front can resolve waiters at the back; push only on the first pass so each index enters once.


## When to use it

Use a monotonic stack when each position needs the next greater (or smaller) value in one direction. Prefer a nested right scan only for tiny lists. Prefer sorting plus a structure when the question is about global order, not nearest neighbor to the right. Prefer a plain running maximum when you only need the best so far, not per-index next greater.


## Brute force, and why it's slow

From each index, scan right until a strictly greater value appears.


In [1]:
def next_greater_slow(nums):
    comparisons = 0
    answer = [-1] * len(nums)
    for index, value in enumerate(nums):
        for right in range(index + 1, len(nums)):
            comparisons += 1
            if nums[right] > value:
                answer[index] = nums[right]
                break
    return answer, comparisons


for size in (50, 100, 200):
    data = list(range(size, 0, -1))
    print(size, next_greater_slow(data)[1])


50 1225
100 4950
200 19900


## Key insight

Keep a stack of indexes waiting for a greater value. When the current value beats the top, pop and fill that answer. Each index enters and leaves the stack at most once.


## Worked example

For each value, return the next strictly greater value to its right, or `-1` when none exists.


In [2]:
def next_greater(nums: list[int]) -> list[int]:
    # 1. Default: nothing greater was found.
    answer = [-1] * len(nums)
    stack: list[int] = []
    for index, value in enumerate(nums):
        # 2. Resolve every waiter smaller than the current value.
        while stack and nums[stack[-1]] < value:
            answer[stack.pop()] = value
        # 3. Current index waits for a later greater value.
        stack.append(index)
    return answer


print(next_greater([2, 1, 3]))
print(next_greater([5, 4]))
print(next_greater([]))


[3, 3, -1]
[-1, -1]
[]


## Trace

`[2, 1, 3]`:

| index | value | stack before | action | answer |
|---|---|---|---|---|
| 0 | 2 | `[]` | push 0 | `[-1, -1, -1]` |
| 1 | 1 | `[0]` | 1 < 2, push 1 | `[-1, -1, -1]` |
| 2 | 3 | `[0, 1]` | pop 1 → 3, pop 0 → 3, push 2 | `[3, 3, -1]` |


## Cost

Each index is pushed and popped at most once: time grows with `n`. Extra memory is the stack and the answer list. The nested right scan pays about `n²` comparisons on a strictly decreasing list.


## Variants

Not solved here: [Daily Temperatures](../../challenges/problems/23-daily-temperatures.ipynb), [Trapping Rain Water](../../challenges/problems/22-trapping-rain-water.ipynb), [Valid Parentheses](../../challenges/problems/08-valid-parentheses.ipynb).


## Common mistake

This bug is silent. The comparison uses `<=`, so an equal value counts as "greater". There is no traceback. The print is `[2, -1]`, and the answer is `[-1, -1]`.


In [3]:
def next_greater_wrong(nums):
    answer = [-1] * len(nums)
    stack = []
    for index, value in enumerate(nums):
        while stack and nums[stack[-1]] <= value:  # equals should not win
            answer[stack.pop()] = value
        stack.append(index)
    return answer


print(next_greater_wrong([2, 2]))


[2, -1]


The fix compares with `<` so only a strictly greater value resolves a waiter. Run `next_greater([2, 2])` and you get `[-1, -1]`.


In [4]:
print(next_greater([2, 2]))


[-1, -1]


## Python idioms

Store indexes, not values, so you can write `answer[index] = value` when a waiter resolves. Initialize the answer with `-1` so unresolved indexes need no extra pass.


In [5]:
def next_greater_verbose(nums: list[int]) -> list[int]:
    answer = [-1] * len(nums)
    waiting: list[int] = []
    for index, value in enumerate(nums):
        while waiting and nums[waiting[-1]] < value:
            earlier = waiting.pop()
            answer[earlier] = value
        waiting.append(index)
    return answer


print(next_greater_verbose([2, 1, 3]))


[3, 3, -1]


## Recap

- Waiting indexes sit in a decreasing-value stack.
- A larger current value resolves the top, possibly many times.
- Use strict `<` when the prompt says strictly greater.

Next: [practice](02-min-stack-practice.ipynb).
